# 03 — Final forecasting & scheduling backtest

**Purpose:** Reproduce the corrected retrospective July–December 2025 evaluation independently of exploratory notebook state.

**Locked specification:** GB (`max_iter=200`, `learning_rate=0.05`, `max_leaf_nodes=15`, `l2_regularization=10`), Ridge (`alpha=30000`), own-price lags 144/192/336/384, cyclical calendar features. Fit labels strictly before 2025-06-29 00:00 UTC. Workload = 6 MWh/day, max = 0.5 MWh/half-hour (1 MW), 24-hour scheduling window. Historical lag 7/14/21 days when forecast missing. No test-set tuning.

**Caveats:** This is a *corrected retrospective* test, not a fully untouched real-time-vintage experiment. Price is the APX market index, not a data-centre retail tariff. Carbon is an average-intensity attribution, not avoided grid emissions. Missing actual outcomes are not imputed.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
PRICE_PATH = DATA / "electricity_price_2024_2025.csv"
CARBON_PATH = DATA / "carbon_intensity_2024_2025.csv"
for path in (PRICE_PATH, CARBON_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Cannot find {path}. Open notebook from project root/notebooks.")

price_raw = pd.read_csv(PRICE_PATH, parse_dates=["startTime"])
carbon_raw = pd.read_csv(CARBON_PATH, parse_dates=["from"])
price_raw["startTime"] = pd.to_datetime(price_raw["startTime"], utc=True)
carbon_raw["from"] = pd.to_datetime(carbon_raw["from"], utc=True)
price_raw = price_raw.set_index("startTime").sort_index()
carbon_raw = carbon_raw.set_index("from").sort_index()

assert price_raw.index.is_unique and carbon_raw.index.is_unique
expected_index = pd.date_range("2024-01-01", "2026-01-01", freq="30min", tz="UTC", inclusive="left")
assert price_raw.index.equals(expected_index), "Price timestamps do not match full UTC grid"
assert carbon_raw.index.equals(expected_index), "Carbon timestamps do not match full UTC grid"

targets = pd.DataFrame({
    "price": price_raw["price"].where(price_raw["volume"] > 0),
    "carbon": carbon_raw["carbon_actual_gco2_kwh"],
}, index=expected_index)
print("Loaded", len(targets), "periods")
print("Missing/invalid price:", targets["price"].isna().sum())
print("Missing carbon:", targets["carbon"].isna().sum())

Loaded 35088 periods
Missing/invalid price: 12
Missing carbon: 31


## 1. Forecast features and strictly pre-test training

In [2]:
features = pd.DataFrame(index=targets.index)
for lag in [144, 192, 336, 384]:
    features[f"price_lag_{lag}"] = targets["price"].shift(lag)

hour = targets.index.hour + targets.index.minute / 60
weekday = targets.index.dayofweek
day_of_year = targets.index.dayofyear
for name, value, period in [
    ("hour", hour, 24),
    ("weekday", weekday, 7),
    ("year", day_of_year, 365.25),
]:
    features[f"{name}_sin"] = np.sin(2 * np.pi * value / period)
    features[f"{name}_cos"] = np.cos(2 * np.pi * value / period)

fit_cutoff = pd.Timestamp("2025-06-29", tz="UTC")
test_start = pd.Timestamp("2025-07-01", tz="UTC")
test_end = pd.Timestamp("2026-01-01", tz="UTC")
fit_mask = (targets.index < fit_cutoff) & features.notna().all(axis=1) & targets["price"].notna()
test_mask = (targets.index >= test_start) & (targets.index < test_end)

# All 3-day+ target-relative lags are available under the assumed 24h delay.
assert min([144, 192, 336, 384]) >= 96
assert targets.index[fit_mask].max() < fit_cutoff

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=30000)),
    "Gradient Boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=10,
        early_stopping=False,
        random_state=42,
    ),
}
for model in models.values():
    model.fit(features.loc[fit_mask], targets.loc[fit_mask, "price"])

forecast_df = targets.loc[test_mask].copy()
valid_inputs = features.loc[test_mask].notna().all(axis=1)
for name, model in models.items():
    forecast_df[name] = np.nan
    forecast_df.loc[valid_inputs, name] = model.predict(features.loc[test_mask].loc[valid_inputs])

forecast_df["7-day naive"] = targets["price"].shift(336).loc[test_mask]

# Predetermined fallback hierarchy using only historical prices.
# Uses target-time lags of 7, 14 and 21 days; does not fill actual outcomes.
fallback = targets["price"].shift(336)
for lag in [672, 1008]:
    fallback = fallback.fillna(targets["price"].shift(lag))

names = ["7-day naive", "Ridge", "Gradient Boosting"]
fallback_usage = {}
for name in names:
    missing_before = forecast_df[name].isna()
    forecast_df[name] = forecast_df[name].fillna(fallback.reindex(forecast_df.index))
    fallback_usage[name] = int((missing_before & forecast_df[name].notna()).sum())

print("Training rows:", int(fit_mask.sum()))
print("Fallback observations (entire test):", fallback_usage)

Training rows: 25764
Fallback observations (entire test): {'7-day naive': 11, 'Ridge': 41, 'Gradient Boosting': 41}


## 2. Full-day coverage and constrained optimisation

In [3]:
REQUIRED = ["price", "carbon", "7-day naive", "Ridge", "Gradient Boosting"]
coverage_daily = forecast_df[REQUIRED].groupby(forecast_df.index.floor("D")).apply(
    lambda g: len(g) == 48 and g.notna().all().all()
)
print("Total test days:", len(coverage_daily))
print("Complete days:", int(coverage_daily.sum()))
print("Excluded days:", list(coverage_daily.index[~coverage_daily]))

ENERGY_MWH = 6.0
MAX_SLOT_MWH = 0.5

def optimise(predicted_price):
    p = np.asarray(predicted_price, dtype=float)
    if len(p) != 48 or not np.isfinite(p).all():
        raise ValueError("Require 48 finite price forecasts")
    result = linprog(
        c=p,
        A_eq=np.ones((1, 48)),
        b_eq=[ENERGY_MWH],
        bounds=(0, MAX_SLOT_MWH),
        method="highs",
    )
    if not result.success:
        raise RuntimeError(result.message)
    assert np.isclose(result.x.sum(), ENERGY_MWH)
    assert result.x.min() >= -1e-7 and result.x.max() <= MAX_SLOT_MWH + 1e-7
    return result.x

records = []
for day, group in forecast_df[REQUIRED].groupby(forecast_df.index.floor("D")):
    if not coverage_daily.loc[day]:
        continue
    realised_price = group["price"].to_numpy()
    realised_carbon = group["carbon"].to_numpy()
    schedules = {
        "Uniform": np.full(48, ENERGY_MWH / 48),
        "Fixed": np.r_[np.full(12, MAX_SLOT_MWH), np.zeros(36)],
        "7-day naive": optimise(group["7-day naive"].to_numpy()),
        "Ridge": optimise(group["Ridge"].to_numpy()),
        "Gradient Boosting": optimise(group["Gradient Boosting"].to_numpy()),
        "Oracle": optimise(realised_price),  # perfect-information bound only
    }
    for policy, x in schedules.items():
        records.append({
            "date": day,
            "policy": policy,
            "cost_gbp": realised_price @ x,
            # g/kWh × MWh = kg of CO2 (1 MWh=1000 kWh; 1000g=1kg)
            "carbon_kg": realised_carbon @ x,
        })

final_test_results = pd.DataFrame(records)
final_summary = final_test_results.groupby("policy").agg(
    days=("date", "nunique"),
    cost_gbp=("cost_gbp", "sum"),
    carbon_kg=("carbon_kg", "sum"),
)
reference = final_summary.loc["Uniform"]
final_summary["cost_saving_pct"] = 100 * (1 - final_summary.cost_gbp / reference.cost_gbp)
final_summary["carbon_reduction_pct"] = 100 * (1 - final_summary.carbon_kg / reference.carbon_kg)
print(final_summary.round(3).to_string())
print("\nBenchmark GB savings vs Uniform: ~20.456% (allow small library-version differences)")

Total test days: 184
Complete days: 183
Excluded days: [Timestamp('2025-09-06 00:00:00+0000', tz='UTC')]
                   days   cost_gbp  carbon_kg  cost_saving_pct  carbon_reduction_pct
policy                                                                              
7-day naive         183  67975.755  116858.50           15.840                15.072
Fixed               183  69032.790  124670.00           14.531                 9.395
Gradient Boosting   183  64247.330  112798.00           20.456                18.023
Oracle              183  57092.730  110814.50           29.314                19.465
Ridge               183  66054.285  116840.50           18.219                15.085
Uniform             183  80769.279  137597.75            0.000                 0.000

Benchmark GB savings vs Uniform: ~20.456% (allow small library-version differences)


## 3. Prediction accuracy, daily ranking and fallback coverage

In [4]:
eligible_dates = final_test_results["date"].unique()
evaluation = forecast_df.loc[forecast_df.index.floor("D").isin(eligible_dates)]
accuracy = []
for name in names:
    rho = [spearmanr(g["price"], g[name]).statistic
           for _, g in evaluation.groupby(evaluation.index.floor("D"))]
    accuracy.append({
        "model": name,
        "observations": len(evaluation),
        "MAE": mean_absolute_error(evaluation["price"], evaluation[name]),
        "RMSE": np.sqrt(mean_squared_error(evaluation["price"], evaluation[name])),
        "daily_spearman": np.nanmean(rho),
    })
accuracy = pd.DataFrame(accuracy).set_index("model")
print(accuracy.round(3).to_string())
print("\nFallback usage on evaluated days:")
for name in names:
    direct_missing = (features.loc[evaluation.index].isna().any(axis=1)
                      if name != "7-day naive" else targets["price"].shift(336).loc[evaluation.index].isna())
    print(f"{name}: {int(direct_missing.sum())} half-hours")

                   observations     MAE    RMSE  daily_spearman
model                                                          
7-day naive                8784  24.149  35.274           0.553
Ridge                      8784  17.548  26.168           0.626
Gradient Boosting          8784  19.386  26.941           0.677

Fallback usage on evaluated days:
7-day naive: 11 half-hours
Ridge: 41 half-hours
Gradient Boosting: 41 half-hours


## 4. Seven-day paired circular moving-block bootstrap

In [5]:
daily = final_test_results.pivot(index="date", columns="policy", values=["cost_gbp", "carbon_kg"]).sort_index()
n = len(daily)
rng = np.random.default_rng(42)
block = 7
n_boot = 5000
start = rng.integers(0, n, size=(n_boot, int(np.ceil(n / block))))
indices = ((start[:, :, None] + np.arange(block)) % n).reshape(n_boot, -1)[:, :n]
print("Evaluation days:", n)
for metric, benchmark in [
    ("cost_gbp", "Uniform"), ("cost_gbp", "Ridge"),
    ("cost_gbp", "7-day naive"), ("carbon_kg", "Uniform"),
]:
    gb = daily[metric]["Gradient Boosting"].to_numpy()
    ref = daily[metric][benchmark].to_numpy()
    point = 100 * (1 - gb.sum() / ref.sum())
    draws = 100 * (1 - gb[indices].sum(axis=1) / ref[indices].sum(axis=1))
    low, high = np.percentile(draws, [2.5, 97.5])
    print(f"{metric} vs {benchmark}: {point:.2f}% [{low:.2f}%, {high:.2f}%]")

Evaluation days: 183
cost_gbp vs Uniform: 20.46% [16.66%, 24.91%]
cost_gbp vs Ridge: 2.74% [0.53%, 5.42%]
cost_gbp vs 7-day naive: 5.48% [2.92%, 8.55%]
carbon_kg vs Uniform: 18.02% [15.04%, 21.17%]


## 5. Interpretation and audit boundaries

- The models were selected using **April–June 2025 validation**; July–December 2025 numbers are **corrected retrospective holdout** results, not a fresh untouched test after the fallback correction.
- The 24-hour observation delay is **assumed**. Actual Elexon/API release timestamps and archived forecast vintages have not been verified.
- The baseline is **6 MWh spread uniformly over 48 half-hours**, not a real facility bill. The wholesale market index excludes network charges, retail fees, demand charges and taxes.
- Scheduling is a **divisible aggregate workload** available from midnight with 1 MW power capacity; it does not model separate job arrivals, deadlines, performance costs or peak fees.
- Carbon results use **realised national average carbon intensity** as an attribution proxy, not marginal/causally avoided grid emissions.
- Days lacking actual outcomes are excluded, not imputed. Results refer to eligible evaluation days.
- If a source dataset, library version or data cleaning changes, rerun and record resulting differences; do not silently preserve the benchmark numbers.